In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "runs" / "models"
REPORT_DIR = PROJECT_ROOT / "runs" / "reports"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
validation_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
retrieval_train_df = pd.read_csv(PROCESSED_DIR / "retrieval_train.csv")

user_mapping = pd.read_csv(PROCESSED_DIR / "user_mapping.csv")
movie_mapping = pd.read_csv(PROCESSED_DIR / "movie_mapping.csv")

user_embeddings = np.load(MODEL_DIR / "user_embeddings.npy")
movie_embeddings = np.load(MODEL_DIR / "movie_embeddings.npy")

print("Data loaded successfully.")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)
print("Retrieval train:", retrieval_train_df.shape)

print("User embeddings:", user_embeddings.shape)
print("Movie embeddings:", movie_embeddings.shape)

Data loaded successfully.
Train: (80419, 6)
Validation: (9818, 6)
Test: (10599, 6)
Retrieval train: (39071, 7)
User embeddings: (610, 64)
Movie embeddings: (9742, 64)


Inspect the mapping columns

In [7]:
print("User mapping columns:")
print(user_mapping.columns.tolist())

display(user_mapping.head())

print("\nMovie mapping columns:")
print(movie_mapping.columns.tolist())

display(movie_mapping.head())

User mapping columns:
['userId', 'user_idx']


,userId,user_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4



Movie mapping columns:
['movieId', 'movie_idx']


,movieId,movie_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


Check embedding dimensions and index ranges

In [8]:
print("Number of users in mapping:", len(user_mapping))
print("Number of movies in mapping:", len(movie_mapping))

print("User embedding rows:", user_embeddings.shape[0])
print("Movie embedding rows:", movie_embeddings.shape[0])

print("\nUser index range:")
print(user_mapping["user_idx"].min(), "to", user_mapping["user_idx"].max())

print("\nMovie index range:")
print(movie_mapping["movie_idx"].min(), "to", movie_mapping["movie_idx"].max())

Number of users in mapping: 610
Number of movies in mapping: 9742
User embedding rows: 610
Movie embedding rows: 9742

User index range:
0 to 609

Movie index range:
0 to 9741


Check mapping uniqueness

In [9]:
print("Duplicate user indices:",
      user_mapping["user_idx"].duplicated().sum())

print("Duplicate movie indices:",
      movie_mapping["movie_idx"].duplicated().sum())

print("Duplicate original user IDs:",
      user_mapping.iloc[:, 0].duplicated().sum())

print("Duplicate original movie IDs:",
      movie_mapping.iloc[:, 0].duplicated().sum())

Duplicate user indices: 0
Duplicate movie indices: 0
Duplicate original user IDs: 0
Duplicate original movie IDs: 0


Check index ranges in datasets

In [10]:
for name, df in {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df,
    "Retrieval Train": retrieval_train_df
}.items():

    print(f"\n{name} dataset")

    print("User index min:", df["user_idx"].min())
    print("User index max:", df["user_idx"].max())

    print("Movie index min:", df["movie_idx"].min())
    print("Movie index max:", df["movie_idx"].max())

    invalid_users = (
        (df["user_idx"] < 0) |
        (df["user_idx"] >= user_embeddings.shape[0])
    ).sum()

    invalid_movies = (
        (df["movie_idx"] < 0) |
        (df["movie_idx"] >= movie_embeddings.shape[0])
    ).sum()

    print("Invalid user indices:", invalid_users)
    print("Invalid movie indices:", invalid_movies)


Train dataset
User index min: 0
User index max: 609
Movie index min: 0
Movie index max: 9741
Invalid user indices: 0
Invalid movie indices: 0

Validation dataset
User index min: 0
User index max: 609
Movie index min: 0
Movie index max: 9738
Invalid user indices: 0
Invalid movie indices: 0

Test dataset
User index min: 0
User index max: 609
Movie index min: 0
Movie index max: 9740
Invalid user indices: 0
Invalid movie indices: 0

Retrieval Train dataset
User index min: 0
User index max: 609
Movie index min: 0
Movie index max: 9741
Invalid user indices: 0
Invalid movie indices: 0


Check how many movies were actually trained

In [11]:
trained_movie_indices = set(
    retrieval_train_df["movie_idx"].unique()
)

all_movie_indices = set(
    range(movie_embeddings.shape[0])
)

untrained_movie_indices = (
    all_movie_indices - trained_movie_indices
)

print("Movies with positive training interactions:",
      len(trained_movie_indices))

print("Movies without positive training interactions:",
      len(untrained_movie_indices))

print("Total movies:", len(all_movie_indices))

print(
    "Percentage of movies without positive training interactions:",
    round(
        len(untrained_movie_indices) / len(all_movie_indices) * 100,
        2
    ),
    "%"
)

Movies with positive training interactions: 5320
Movies without positive training interactions: 4422
Total movies: 9742
Percentage of movies without positive training interactions: 45.39 %


Check embedding norms

In [12]:
user_norms = np.linalg.norm(user_embeddings, axis=1)
movie_norms = np.linalg.norm(movie_embeddings, axis=1)

print("User embedding norms")
print("Mean:", user_norms.mean())
print("Min:", user_norms.min())
print("Max:", user_norms.max())

print("\nMovie embedding norms")
print("Mean:", movie_norms.mean())
print("Min:", movie_norms.min())
print("Max:", movie_norms.max())

User embedding norms
Mean: 2.8904846
Min: 1.1785737
Max: 6.225627

Movie embedding norms
Mean: 1.490018
Min: 0.6932865
Max: 4.0806403


Check recommendation score distribution

In [13]:
sample_user_idx = int(test_df["user_idx"].iloc[0])

sample_user_embedding = user_embeddings[sample_user_idx]

sample_scores = np.dot(
    movie_embeddings,
    sample_user_embedding
)

print("Sample user index:", sample_user_idx)

print("Score minimum:", sample_scores.min())
print("Score maximum:", sample_scores.max())
print("Score mean:", sample_scores.mean())
print("Score standard deviation:", sample_scores.std())

top_indices = np.argsort(sample_scores)[::-1][:10]

print("\nTop 10 movie indices:")
print(top_indices)

Sample user index: 0
Score minimum: -9.850212
Score maximum: -0.99524724
Score mean: -2.5780861
Score standard deviation: 1.129746

Top 10 movie indices:
[2173 7480 1192 2991 2684 1288 2770 2652  768 1125]


Check test relevance and training overlap

In [14]:
RELEVANCE_THRESHOLD = 4.0

test_positive_df = test_df[
    test_df["rating"] >= RELEVANCE_THRESHOLD
].copy()

train_interactions = set(
    zip(train_df["user_idx"], train_df["movie_idx"])
)

test_positive_interactions = set(
    zip(test_positive_df["user_idx"], test_positive_df["movie_idx"])
)

overlap = train_interactions & test_positive_interactions

print("Positive test interactions:", len(test_positive_interactions))

print("Overlap with training interactions:", len(overlap))

print("Unique users in positive test set:",
      test_positive_df["user_idx"].nunique())

print("Unique relevant test movies:",
      test_positive_df["movie_idx"].nunique())

Positive test interactions: 5058
Overlap with training interactions: 0
Unique users in positive test set: 574
Unique relevant test movies: 2626


Check if positive test movies have trained embeddings

In [15]:
test_positive_movie_indices = set(
    test_positive_df["movie_idx"].unique()
)

positive_test_movies_without_training = (
    test_positive_movie_indices - trained_movie_indices
)

print("Positive test movies:",
      len(test_positive_movie_indices))

print("Positive test movies without retrieval training:",
      len(positive_test_movies_without_training))

print(
    "Percentage of positive test movies without retrieval training:",
    round(
        len(positive_test_movies_without_training) /
        max(len(test_positive_movie_indices), 1) * 100,
        2
    ),
    "%"
)

Positive test movies: 2626
Positive test movies without retrieval training: 581
Percentage of positive test movies without retrieval training: 22.12 %


Generate a diagnostic summary

In [16]:
diagnostics = {
    "user_embedding_count": user_embeddings.shape[0],
    "movie_embedding_count": movie_embeddings.shape[0],
    "trained_movie_count": len(trained_movie_indices),
    "untrained_movie_count": len(untrained_movie_indices),
    "positive_test_interactions": len(test_positive_df),
    "positive_test_movies": len(test_positive_movie_indices),
    "positive_test_movies_without_training": len(
        positive_test_movies_without_training
    ),
    "train_test_overlap": len(overlap),
    "user_embedding_mean_norm": float(user_norms.mean()),
    "movie_embedding_mean_norm": float(movie_norms.mean())
}

diagnostics_df = pd.DataFrame(
    diagnostics.items(),
    columns=["Diagnostic", "Value"]
)

display(diagnostics_df)

diagnostics_df.to_csv(
    REPORT_DIR / "two_tower_diagnostics.csv",
    index=False
)

print("Diagnostic report saved.")

,Diagnostic,Value
0,user_embedding_count,610.000000
1,movie_embedding_count,9742.000000
2,trained_movie_count,5320.000000
3,untrained_movie_count,4422.000000
4,positive_test_interactions,5058.000000
5,positive_test_movies,2626.000000
6,positive_test_movies_without_training,581.000000
7,train_test_overlap,0.000000
8,user_embedding_mean_norm,2.890485
9,movie_embedding_mean_norm,1.490018


Diagnostic report saved.
